In [1]:
import os
os.chdir("..")
import time
import pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression

In [2]:
FEATURE_COLS = ["hr", "hrv", "eda", "wrist_temp", "co2_noisy", "lux_noisy", "posture_cm"]
df = pd.read_csv("outputs/unified_features.csv").dropna(subset=FEATURE_COLS)

le = LabelEncoder()
y = le.fit_transform(df["ground_truth"])
X = df[FEATURE_COLS].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

MODELS = {
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "SVM": SVC(kernel="rbf", class_weight="balanced"),
    "RandomForest": RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=42),
    "GradientBoosting": GradientBoostingClassifier(random_state=42),
    "MLP": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42),
}

deployability = []
sample_batch = X_scaled[:100]

for name, model in MODELS.items():
    model.fit(X_scaled, y)

    with open("temp_model.pkl", "wb") as f:
        pickle.dump(model, f)
    size_kb = os.path.getsize("temp_model.pkl") / 1024
    os.remove("temp_model.pkl")

    times = []
    for _ in range(20):
        start = time.perf_counter()
        model.predict(sample_batch)
        times.append((time.perf_counter() - start) / len(sample_batch))
    avg_latency_ms = np.mean(times) * 1000

    deployability.append({
        "Model": name, "Size_KB": round(size_kb, 1),
        "Latency_ms_per_sample": round(avg_latency_ms, 4)
    })
    print(f"{name}: {size_kb:.1f} KB, {avg_latency_ms:.4f} ms/prediction")

LogisticRegression: 0.8 KB, 0.0031 ms/prediction
SVM: 38.3 KB, 0.0844 ms/prediction
RandomForest: 2334.3 KB, 0.3671 ms/prediction
GradientBoosting: 133.1 KB, 0.0107 ms/prediction
MLP: 93.6 KB, 0.0116 ms/prediction


In [3]:
deploy_df = pd.DataFrame(deployability).sort_values("Size_KB")
deploy_df.to_csv("outputs/edge_deployability_results.csv", index=False)
print(deploy_df.to_string(index=False))

             Model  Size_KB  Latency_ms_per_sample
LogisticRegression      0.8                 0.0031
               SVM     38.3                 0.0844
               MLP     93.6                 0.0116
  GradientBoosting    133.1                 0.0107
      RandomForest   2334.3                 0.3671
